# Perform data loading from CSV files to Google Cloud SQL tables

In [1]:
# import packages that will be used

import io
import os
from typing import Dict

import pandas as pd
from dotenv import load_dotenv
from google.cloud import storage
from google.cloud.sql.connector import Connector
from sqlalchemy import create_engine, text
from pathlib import Path

In [6]:
# ------------------------------------------------------------
# Load environment variables
# ------------------------------------------------------------
env_path = Path.cwd().parent / "src/.env"
if not env_path.exists():
    raise FileNotFoundError(f".env file not found: {env_path}")

load_dotenv(dotenv_path=env_path, override=True)

INSTANCE_CONNECTION_NAME = os.getenv("INSTANCE_CONNECTION_NAME")
DB_USER = os.getenv("DB_USER")
DB_PASS = os.getenv("DB_PASS")
DB_NAME = os.getenv("DB_NAME")

GCS_BUCKET_NAME = os.getenv("GCS_BUCKET_NAME")
GCS_PREFIX = os.getenv("GCS_PREFIX")
BATCH_TIME = os.getenv("BATCH_TIME")   

POSTGRES_SCHEMA = os.getenv("POSTGRES_SCHEMA")


required_vars = {
    "INSTANCE_CONNECTION_NAME": INSTANCE_CONNECTION_NAME,
    "DB_USER": DB_USER,
    "DB_PASS": DB_PASS,
    "DB_NAME": DB_NAME,
    "GCS_BUCKET_NAME": GCS_BUCKET_NAME,
    "GCS_PREFIX": GCS_PREFIX,
    "BATCH_TIME": BATCH_TIME,
    "POSTGRES_SCHEMA": POSTGRES_SCHEMA
}

missing_vars = [key for key, value in required_vars.items() if not value]

if missing_vars:
    raise ValueError(f"Missing required .env values: {missing_vars}")
else:
    print("All required environment variables are set.\n")
    

print(f"GCP_PROJECT_ID: {os.getenv('GCP_PROJECT_ID')}")
print(f"Using Cloud SQL instance: {INSTANCE_CONNECTION_NAME}")
print(f"Using GCS bucket: {GCS_BUCKET_NAME} with prefix: {GCS_PREFIX}")
print(f"Batch time for ingestion metadata: {BATCH_TIME}")
print(f"Database user: {DB_USER}")
print(f"Database name: {DB_NAME}")
print(f"Database password: {'*' * len(DB_PASS)}")  # Mask the password in logs
print(f"Using PostgreSQL schema: {POSTGRES_SCHEMA}")


All required environment variables are set.

GCP_PROJECT_ID: sctp-team2-project2-elt
Using Cloud SQL instance: sctp-team2-project2-elt:us-central1:sctp-m2-olist
Using GCS bucket: m2_olin_raw with prefix: [DONE] Initial Load/
Batch time for ingestion metadata: 2026-06-07T16:30:00Z
Database user: postgres
Database name: olist
Database password: ****************
Using PostgreSQL schema: oltp


In [ ]:
# ------------------------------------------------------------
# CSV file to PostgreSQL table mapping
# ------------------------------------------------------------
from duckdb import df


CSV_TO_TABLE: Dict[str, str] = {
    # DONE "olist_customers_dataset.csv": "olist_customers",
    # "olist_geolocation_dataset.csv": "olist_geolocation",
    # DONE "olist_order_items_dataset.csv": "olist_order_items",
    # DONE "olist_order_payments_dataset.csv": "olist_order_payments",
    # DONE "olist_order_reviews_dataset.csv": "olist_order_reviews",
    # DONE "olist_orders_dataset.csv": "olist_orders",
    # DONE "olist_products_dataset.csv": "olist_products",
    # DONE "olist_sellers_dataset.csv": "olist_sellers",
    # DONE "product_category_name_translation.csv": "product_category_name_translation"
}


# ------------------------------------------------------------
# Cloud SQL PostgreSQL connection
# ------------------------------------------------------------
connector = Connector()


def getconn():
    return connector.connect(
        INSTANCE_CONNECTION_NAME,
        "pg8000",
        user=DB_USER,
        password=DB_PASS,
        db=DB_NAME,
    )


engine = create_engine(
    "postgresql+pg8000://",
    creator=getconn,
)


# ------------------------------------------------------------
# Create schema if it does not exist
# ------------------------------------------------------------
def create_schema_if_not_exists() -> None:
    with engine.begin() as conn:
        conn.execute(text(f'CREATE SCHEMA IF NOT EXISTS "{POSTGRES_SCHEMA}"'))
    print(f"Schema ready: {POSTGRES_SCHEMA}")


# ------------------------------------------------------------
# Load one CSV from GCS into PostgreSQL
# ------------------------------------------------------------
def load_csv_from_gcs_to_postgres(
    storage_client: storage.Client,
    filename: str,
    table_name: str,
    if_exists: str = "replace",
) -> None:
    bucket = storage_client.bucket(GCS_BUCKET_NAME)
    blob_path = f"{GCS_PREFIX.rstrip('/')}/{filename}"
    blob = bucket.blob(blob_path)

    if not blob.exists():
        raise FileNotFoundError(f"File not found: gs://{GCS_BUCKET_NAME}/{blob_path}")

    gsfilepath=f"gs://{GCS_BUCKET_NAME}/{blob_path}"
    print(f"\nReading {gsfilepath}")

    csv_bytes = blob.download_as_bytes()
    df = pd.read_csv(io.BytesIO(csv_bytes))

    # Add ingestion metadata
    #now_utc = pd.Timestamp.now(tz="UTC")
    # set specific timestamp for testing
    # now_utc = pd.Timestamp("2026-06-07T16:30:00Z")
    now_utc = pd.Timestamp(BATCH_TIME)
    
    print(f"Current/Batch UTC time: {now_utc}")
    
    df["created_at"] = now_utc
    df["updated_at"] = now_utc
    df["is_deleted"] = False
    df["source_file"] = filename
    df["source_gcs_path"] = gsfilepath
    df["batch_name"] = GCS_PREFIX.strip("/").split("/")[-1]  # Use last part of prefix as batch name
    
    print(f"Loading {len(df):,} rows into {POSTGRES_SCHEMA}.{table_name}")

    df.to_sql(
        table_name,
        engine,
        schema=POSTGRES_SCHEMA,
        if_exists=if_exists,
        index=False,
        chunksize=1_000,
        method="multi",
    )

    print(f"Loaded {POSTGRES_SCHEMA}.{table_name}")
    
    with engine.begin() as conn:
        print(f"Analyzing {POSTGRES_SCHEMA}.{table_name} for query performance...")
        conn.execute(text(f'ANALYZE "{POSTGRES_SCHEMA}"."{table_name}"'))
        # 3. PostgreSQL table-level statistics
        table_stats_sql = text("""
            SELECT
                schemaname,
                relname AS table_name,
                n_live_tup AS estimated_live_rows,
                n_dead_tup AS estimated_dead_rows,
                last_analyze,
                last_autoanalyze
            FROM pg_stat_user_tables
            WHERE schemaname = :schema_name
              AND relname = :table_name
        """)

        table_stats = pd.read_sql(
            table_stats_sql,
            conn,
            params={
                "schema_name": POSTGRES_SCHEMA,
                "table_name": table_name
            }
        )

        print("\nPostgreSQL table statistics:")
        print(table_stats)
    


# ------------------------------------------------------------
# Main
# ------------------------------------------------------------
def main() -> None:
    create_schema_if_not_exists()

    storage_client = storage.Client()

    for filename, table_name in CSV_TO_TABLE.items():
        print(f"\nProcessing file: {filename} -> table: {table_name}")
        load_csv_from_gcs_to_postgres(
            storage_client=storage_client,
            filename=filename,
            table_name=table_name,
            # if_exists="replace",  # use "append" after initial load
            if_exists="append",  # use "append" after initial load            
        )

    connector.close()
    print("\nAll CSV files loaded successfully.")


if __name__ == "__main__":
    main()

Schema ready: oltp

Processing file: olist_geolocation_dataset.csv -> table: olist_geolocation

Reading gs://m2_olin_raw/[DONE] Initial Load/olist_geolocation_dataset.csv
Current/Batch UTC time: 2026-06-07 16:30:00+00:00
Loading 1,000,163 rows into oltp.olist_geolocation
Loaded oltp.olist_geolocation
Analyzing oltp.olist_geolocation for query performance...

PostgreSQL table statistics:
  schemaname         table_name  estimated_live_rows  estimated_dead_rows  \
0       oltp  olist_geolocation              1000163                    0   

                      last_analyze last_autoanalyze  
0 2026-06-08 14:40:23.255911+00:00             None  

All CSV files loaded successfully.
